# 12: LGBM sample weight search

Rol: probar sample weights solo para LGBM, usando la mejor combinacion window/cap elegida en el notebook 10.

No usa test oficial. No se elige automaticamente solo por RMSE: se reportan RMSE, MAE, C-MAPSS score y dangerous_error_pct.


In [12]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CMAPSSData").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [13]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)
from src.fd001_experiment_utils import (
    add_bin_metric_columns,
    available_boosting_factories,
    fit_predict_model,
    lgbm_factory,
    metric_row_from_predictions,
    selection_sort,
)


In [14]:
from src.fd001_experiment_utils import weights_from_scheme

WINDOW_CAP_PATH = RESULTS_DIR / "fd001_lgbm_window_cap_search.csv"
OUTPUT_PATH = RESULTS_DIR / "fd001_lgbm_sample_weight_search.csv"

SCHEMES = {
    "none": None,
    "bin_weights": {0: 4.0, 30: 2.0, 60: 1.5, 90: 1.0},
    "aggressive": {0: 6.0, 30: 3.0, 60: 1.5, 90: 1.0},
    "soft": {0: 2.0, 30: 1.5, 60: 1.2, 90: 1.0},
}


In [15]:
if not WINDOW_CAP_PATH.exists():
    raise FileNotFoundError("Primero ejecutar 10_fd001_window_and_cap_search.ipynb para generar fd001_lgbm_window_cap_search.csv")

window_cap = selection_sort(pd.read_csv(WINDOW_CAP_PATH))
best = window_cap.iloc[0]
best_window = int(best["window_size"])
best_cap = int(best["rul_cap"])
print("Mejor combinacion LGBM por validacion artificial:")
display(best[["model_name", "representation", "window_size", "rul_cap", "mae", "rmse", "cmapss_score", "dangerous_error_pct"]])


Mejor combinacion LGBM por validacion artificial:


model_name             LGBMRegressor
representation          temporal_w50
window_size                       50
rul_cap                          125
mae                        11.297658
rmse                       14.428233
cmapss_score              261.005416
dangerous_error_pct         6.060606
Name: 0, dtype: object

In [16]:
base_prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=VALIDATION_RANDOM_STATE,
    max_rul=None,
    cut_ruls=CUT_RULS,
    window_size=best_window,
)
prepared = dict(base_prepared)
prepared["y_train"] = base_prepared["y_train_raw"].clip(upper=best_cap).copy()
representation = f"temporal_w{best_window}"

rows = []
prediction_tables = []
for scheme_name, scheme in SCHEMES.items():
    print(f"Entrenando LGBMRegressor sample_weight_scheme={scheme_name}")
    weights = weights_from_scheme(base_prepared["y_train_raw"], scheme)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pred_df = fit_predict_model(
            prepared,
            lgbm_factory(random_state=VALIDATION_RANDOM_STATE),
            "LGBMRegressor",
            representation,
            sample_weight=weights,
        )
    pred_df["sample_weight_scheme"] = scheme_name
    prediction_tables.append(pred_df)
    row, _ = metric_row_from_predictions(
        pred_df,
        extra={
            "model_name": "LGBMRegressor",
            "representation": representation,
            "window_size": best_window,
            "rul_cap": best_cap,
            "sample_weight_scheme": scheme_name,
            "target_used_for_training": "RUL_capped",
            "n_features": len(prepared["feature_columns"]),
        },
    )
    rows.append(row)

sample_weight_results = selection_sort(pd.DataFrame(rows))
sample_weight_results.to_csv(OUTPUT_PATH, index=False)
pd.concat(prediction_tables, ignore_index=True).to_csv(RESULTS_DIR / "fd001_lgbm_sample_weight_predictions.csv", index=False)
display(sample_weight_results)
print(OUTPUT_PATH)


Entrenando LGBMRegressor sample_weight_scheme=none
Entrenando LGBMRegressor sample_weight_scheme=bin_weights
Entrenando LGBMRegressor sample_weight_scheme=aggressive
Entrenando LGBMRegressor sample_weight_scheme=soft


,representation,model_name,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,window_size,rul_cap,sample_weight_scheme,target_used_for_training,n_features
0,temporal_w50,LGBMRegressor,99,11.194267,14.037945,0.889380,251.554310,2.540953,7.070707,3.788301,0.0,7.481389,5.0,14.225283,30.0,15.341870,0.0,50,125,bin_weights,RUL_capped,119
1,temporal_w50,LGBMRegressor,99,10.888389,14.046041,0.889252,253.632055,2.561940,7.070707,3.483879,0.0,6.063067,0.0,13.825081,35.0,15.654103,0.0,50,125,aggressive,RUL_capped,119
2,temporal_w50,LGBMRegressor,99,11.061073,14.253374,0.885959,260.274518,2.629036,7.070707,3.727651,0.0,6.673143,0.0,15.202969,35.0,14.947974,0.0,50,125,soft,RUL_capped,119
3,temporal_w50,LGBMRegressor,99,11.297658,14.428233,0.883144,261.005416,2.636418,6.060606,4.269154,0.0,6.543172,0.0,14.934006,30.0,15.475423,0.0,50,125,none,RUL_capped,119


C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\results\fd001_lgbm_sample_weight_search.csv
